In [8]:
!pip install -q -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 16.8 MB/s eta 0:00:00


In [6]:
from google.colab import userdata

OR_API = userdata.get("OR_API")

print("OpenRouter API key loaded:", OR_API is not None)

OpenRouter API key loaded: True


In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=OR_API,
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [30]:
import os

files = [
    "/content/predict.py",
    "/content/match_winner_pipeline.joblib",
    "/content/top_player_pipeline.joblib",
    "/content/afl_match_features_v1.csv",
    "/content/afl_player_features_v1.csv"
]

for file in files:
    print("✓" if os.path.exists(file) else "✗", file)

✓ /content/predict.py
✓ /content/match_winner_pipeline.joblib
✓ /content/top_player_pipeline.joblib
✓ /content/afl_match_features_v1.csv
✓ /content/afl_player_features_v1.csv


In [32]:
from pathlib import Path

predict_path = Path("/content/predict.py")

text = predict_path.read_text()

text = text.replace(
    'MATCH_MODEL_PATH = BASE_DIR / "model_artifacts" / "match_winner_pipeline.joblib"',
    'MATCH_MODEL_PATH = BASE_DIR / "match_winner_pipeline.joblib"'
)

text = text.replace(
    'PLAYER_MODEL_PATH = BASE_DIR / "model_artifacts" / "top_player_pipeline.joblib"',
    'PLAYER_MODEL_PATH = BASE_DIR / "top_player_pipeline.joblib"'
)

predict_path.write_text(text)

print("✓ predict.py paths updated")

✓ predict.py paths updated


In [33]:
import importlib
import predict

importlib.reload(predict)

from predict import predict_match_winner, predict_top_player

print("✓ predict_match_winner loaded")
print("✓ predict_top_player loaded")

✓ predict_match_winner loaded
✓ predict_top_player loaded


In [34]:
print("Match model:", type(predict.match_winner_pipeline))
print("Player model:", type(predict.top_player_pipeline))
print("Match features:", predict.match_features.shape)
print("Player features:", predict.player_features.shape)

Match model: <class 'sklearn.pipeline.Pipeline'>
Player model: <class 'sklearn.pipeline.Pipeline'>
Match features: (7904, 26)
Player features: (274089, 7)


In [43]:
import pandas as pd
import numpy as np

player_features = pd.read_csv("afl_player_features_v1.csv")

In [73]:
import os

print(os.path.exists("/content/afl_retrieval_tools.py"))

True


In [75]:
import zipfile
import os

with zipfile.ZipFile("/content/afl_datasets.zip", "r") as zip_ref:
  zip_ref.extractall("/content")

In [76]:
from afl_retrieval_tools import (
    team_record_tool,
    player_season_stats_tool
)

print("Day 3 retrieval tools imported successfully.")

AFL retrieval tools loaded successfully.
match_features: (7904, 26)
player_seasonal_stats: (25491, 54)
player_info: (2848, 16)
Day 3 retrieval tools imported successfully.


/content/afl_retrieval_tools.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  player_seasonal_stats = pd.read_csv(


In [77]:
team_record_test = team_record_tool.invoke({
    "team": "Geelong",
    "opponent": "Essendon"
})

print(team_record_test)

{'team': 'Geelong Cats', 'opponent': 'Essendon Bombers', 'matches': 63, 'wins': 39, 'losses': 23, 'draws': 1}


In [78]:
player_test = player_season_stats_tool.invoke({
    "player_name": "Patrick Dangerfield"
})

print(player_test)

{'player': 'Patrick Dangerfield', 'player_ids': [43674], 'year': None, 'records': [{'year': 2008, 'team': 'Adelaide Crows', 'games_played': 2, 'kicks': 6.0, 'marks': 4.0, 'handballs': 6.0, 'disposals': 12.0, 'goals': 1.0, 'behinds': 1.0, 'tackles': 2.0, 'inside_50s': 2.0, 'clearances': 1.0, 'clangers': 5.0, 'rebound_50s': 1.0, 'one_percenters': 1.0}, {'year': 2009, 'team': 'Adelaide Crows', 'games_played': 19, 'kicks': 96.0, 'marks': 52.0, 'handballs': 150.0, 'disposals': 246.0, 'goals': 17.0, 'behinds': 17.0, 'tackles': 42.0, 'inside_50s': 30.0, 'clearances': 45.0, 'clangers': 30.0, 'rebound_50s': 4.0, 'one_percenters': 16.0}, {'year': 2009, 'team': 'Adelaide Crows', 'games_played': 2, 'kicks': 10.0, 'marks': 6.0, 'handballs': 17.0, 'disposals': 27.0, 'goals': 4.0, 'behinds': 2.0, 'tackles': 5.0, 'inside_50s': 2.0, 'clearances': 3.0, 'clangers': 1.0, 'rebound_50s': 2.0, 'one_percenters': 1.0}, {'year': 2010, 'team': 'Adelaide Crows', 'games_played': 19, 'kicks': 140.0, 'marks': 52.0, 

# **TASK 1**

### System Architecture





```text
                         USER QUERY
                             │
                             ▼
                       ┌───────────┐
                       │   Router  │
                       │   Node    │
                       └─────┬─────┘
                             │
          ┌──────────┬───────┼───────────┐
          ▼          ▼       ▼           ▼
      FACTUAL    RETRIEVAL PREDICTION OFF-TOPIC
          │          │       │           │
          ▼          ▼       ▼           ▼
    Direct Answer Retrieval Prediction  Refusal
                    Tool       Tool
                       │          │
                       └────┬─────┘
                            ▼
                     ┌─────────────┐
                     │ Validation  │
                     │    Node     │
                     └──────┬──────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │ Response        │
                   │ Formatting Node │
                   └────────┬────────┘
                            │
                            ▼
                      FINAL RESPONSE


The factual branch can directly proceed to response formatting because it does not depend on a prediction or retrieval tool.

The retrieval and prediction branches pass through validation to check whether the returned result is valid and usable.

The off-topic branch produces a controlled refusal or redirection.

### State Schema

In [2]:
from typing import TypedDict, Literal

class AFLState(TypedDict, total=False):
    # Current user query
    user_query: str

    # Previous conversation messages
    conversation_history: list

    # Intent detected by the router
    intent: Literal[
        "factual",
        "retrieval",
        "prediction",
        "off-topic"
    ]

    # Graph branch selected by the router
    route: str

    # Results returned by retrieval or prediction tools
    tool_results: dict

    # Result of validation after tool execution
    validation_result: dict

    # Used when the system needs clarification
    needs_clarification: bool

    # Question to ask the user when information is missing
    clarification_question: str

    # Final response generated for the user
    final_response: str

## State Fields

| Field | Purpose |
|---|---|
| `user_query` | Stores the current user's question. |
| `conversation_history` | Stores previous messages so the system can support multi-turn conversations. |
| `intent` | Stores the intent detected by the router. |
| `route` | Stores the graph branch selected for the request. |
| `tool_results` | Stores results returned by retrieval or prediction tools. |
| `validation_result` | Stores whether the returned result passed validation. |
| `needs_clarification` | Indicates that the system cannot safely continue because required information is missing or ambiguous. |
| `clarification_question` | Stores the clarification question that should be presented to the user. |
| `final_response` | Stores the final response returned to the user. |

In [3]:
# Verify State Schema

print("AFLState schema created successfully.")

print("\nSupported intents:")
print([
    "factual",
    "retrieval",
    "prediction",
    "off-topic"
])

print("\nGraph branches:")
print([
    "direct_answer",
    "retrieval",
    "prediction",
    "refusal"
])

print("\nState fields:")
print([
    "user_query",
    "conversation_history",
    "intent",
    "route",
    "tool_results",
    "validation_result",
    "needs_clarification",
    "clarification_question",
    "final_response"
])

AFLState schema created successfully.

Supported intents:
['factual', 'retrieval', 'prediction', 'off-topic']

Graph branches:
['direct_answer', 'retrieval', 'prediction', 'refusal']

State fields:
['user_query', 'conversation_history', 'intent', 'route', 'tool_results', 'validation_result', 'needs_clarification', 'clarification_question', 'final_response']


## Intent Categories

### 1. Factual

A factual query asks for general AFL information that does not require a dataset lookup or prediction model.

Example:

> What is a behind in AFL?

Route:

`Router → Direct Answer → Response Formatting`

---

### 2. Retrieval

A retrieval query requires exact information from the AFL datasets.

Examples:

> How many disposals did Ryan Abbott have in 2019?

> What is Geelong's record against Essendon?

Route:

`Router → Retrieval Tool → Validation → Response Formatting`

The system must not invent numerical statistics. If the dataset does not contain the requested information, the system should report that the information is unavailable.

---

### 3. Prediction

A prediction query asks the system to estimate a future or hypothetical AFL outcome using the prediction models.

Examples:

> Who will win Collingwood vs Geelong?

> Who is predicted to be the top player?

Route:

`Router → Prediction Tool → Validation → Response Formatting`

Prediction responses must include the prediction probability or confidence, relevant grounding features, and a statement that the prediction is model-based and not guaranteed.

---

### 4. Off-topic

An off-topic query is outside the AFL domain.

Examples:

> What is the capital of France?

> What will the weather be tomorrow?

Route:

`Router → Refusal → Response Formatting`

The assistant should politely redirect the user to AFL-related questions.

### Example Query Routing

| User Query | Intent | Route |
|---|---|---|
| What is a behind in AFL? | Factual | Direct Answer |
| What are the basic AFL rules? | Factual | Direct Answer |
| How many disposals did Ryan Abbott have in 2019? | Retrieval | Retrieval Tool |
| What is Geelong's record against Essendon? | Retrieval | Retrieval Tool |
| Who will win Collingwood vs Geelong? | Prediction | Prediction Tool |
| Who is predicted to be the top player? | Prediction | Prediction Tool |
| What is the capital of France? | Off-topic | Refusal |
| What is today's weather? | Off-topic | Refusal |

### Prediction Response Requirements

Prediction requests require additional controls because a prediction is not an established fact.

The prediction branch should return:

- Predicted outcome
- Probability or confidence
- Top 2-3 grounding features
- A short explanation of the prediction
- A disclaimer that the result is model-based and not guaranteed

Example response structure:

Prediction: Collingwood Magpies

Probability: 68%

Grounding:
- Recent win rate
- Average score
- Ladder position

Note: This is a model-based probability, not a guaranteed outcome.

### Why Explicit LangGraph Routing?

Explicit routing is used instead of allowing a generic agent to freely decide which action to take.

The AFL system has clearly defined workflows for factual questions, dataset retrieval, prediction, and off-topic requests. A router can classify the request first and LangGraph can then enforce the appropriate branch.

This design provides several benefits:

- Reduces incorrect tool selection.
- Prevents prediction requests from being treated as ordinary factual questions.
- Prevents unsupported questions from reaching AFL tools.
- Makes the workflow easier to trace and debug.
- Makes routing accuracy easier to evaluate.
- Provides controlled validation for retrieval and prediction results.
- Allows clarification when required information is missing.
- Makes prediction disclaimers and grounding requirements easier to enforce.

This is particularly important for prediction requests because predictions should be presented as model-based estimates rather than guaranteed facts.

# TASK 2

### Build Router Node

In [10]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class RouterOutput(BaseModel):
    intent: Literal[
        "factual",
        "retrieval",
        "prediction",
        "off-topic"
    ] = Field(description="The intent of the user's AFL query.")

router_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an intent classifier for an AFL-only assistant.

Classify the user's query into exactly ONE of these intents:

1. prediction
Use prediction when the user asks for a predicted or future outcome.
Examples:
- Who will win X vs Y?
- Who will win the match?
- Who will top-score?
- Who is likely to be the top player?
- Predict the winner of this match.

2. retrieval
Use retrieval when the user asks for exact AFL statistics, records,
match data, player data, or historical numerical information from the dataset.
Examples:
- What were X's stats last round?
- How many disposals did X have?
- What is X's record against Y?
- How many goals did X score in 2019?

3. factual
Use factual for general AFL knowledge that does not require dataset
retrieval or prediction.
Examples:
- What are the basic AFL rules?
- What is a behind in AFL?
- How many players are on an AFL team?

4. off-topic
Use off-topic for questions unrelated to AFL.
Examples:
- What is the capital of France?
- What is today's weather?
- Tell me a Python joke.

Important:
- Do not classify a statistical question as factual.
- Do not classify a prediction question as retrieval.
- If the query asks for an exact statistic or record, choose retrieval.
- If the query asks who will win, who is likely to win, who will top-score,
  or asks for a prediction, choose prediction.
- If the query is outside AFL, choose off-topic.
"""
    ),
    ("human", "{query}")
])

structured_router = llm.with_structured_output(RouterOutput)

router_chain = router_prompt | structured_router


def router_node(state: AFLState):
    query = state["user_query"]

    result = router_chain.invoke({
        "query": query
    })

    intent = result.intent

    route_mapping = {
        "factual": "direct_answer",
        "retrieval": "retrieval",
        "prediction": "prediction",
        "off-topic": "refusal"
    }

    route = route_mapping[intent]

    return {
        "intent": intent,
        "route": route
    }

In [11]:
routing_tests = [
    {
        "query": "What are the basic rules of AFL?",
        "expected_intent": "factual"
    },
    {
        "query": "What is a behind in AFL?",
        "expected_intent": "factual"
    },
    {
        "query": "How many players are on an AFL team?",
        "expected_intent": "factual"
    },
    {
        "query": "What does a mark mean in AFL?",
        "expected_intent": "factual"
    },
    {
        "query": "How many disposals did Ryan Abbott have in 2019?",
        "expected_intent": "retrieval"
    },
    {
        "query": "What were Ryan Abbott's stats last round?",
        "expected_intent": "retrieval"
    },
    {
        "query": "How many goals did Ryan Abbott score in 2019?",
        "expected_intent": "retrieval"
    },
    {
        "query": "What is Geelong's record against Essendon?",
        "expected_intent": "retrieval"
    },
    {
        "query": "How many matches have Geelong and Essendon played?",
        "expected_intent": "retrieval"
    },
    {
        "query": "How many tackles did Ryan Abbott have?",
        "expected_intent": "retrieval"
    },
    {
        "query": "Who will win Collingwood vs Geelong?",
        "expected_intent": "prediction"
    },
    {
        "query": "Who is likely to win the next AFL match?",
        "expected_intent": "prediction"
    },
    {
        "query": "Who will top-score in the match?",
        "expected_intent": "prediction"
    },
    {
        "query": "Predict the winner of Geelong vs Essendon.",
        "expected_intent": "prediction"
    },
    {
        "query": "Who is predicted to be the top player?",
        "expected_intent": "prediction"
    },
    {
        "query": "What is the capital of France?",
        "expected_intent": "off-topic"
    },
    {
        "query": "What will the weather be tomorrow?",
        "expected_intent": "off-topic"
    },
    {
        "query": "Write a Python program for me.",
        "expected_intent": "off-topic"
    },
    {
        "query": "Who is the best basketball player?",
        "expected_intent": "off-topic"
    },
    {
        "query": "Tell me about today's cricket match.",
        "expected_intent": "off-topic"
    }
]

#### Testing Routing

In [12]:
routing_results = []

for test in routing_tests:
    result = router_node({
        "user_query": test["query"]
    })

    actual_intent = result["intent"]

    routing_results.append({
        "Query": test["query"],
        "Expected": test["expected_intent"],
        "Actual": actual_intent,
        "Pass": actual_intent == test["expected_intent"]
    })

for i, result in enumerate(routing_results, 1):
    print(f"{i}. {result['Query']}")
    print(f"   Expected: {result['Expected']}")
    print(f"   Actual:   {result['Actual']}")
    print(f"   Result:   {'PASS' if result['Pass'] else 'FAIL'}")
    print()

1. What are the basic rules of AFL?
   Expected: factual
   Actual:   factual
   Result:   PASS

2. What is a behind in AFL?
   Expected: factual
   Actual:   factual
   Result:   PASS

3. How many players are on an AFL team?
   Expected: factual
   Actual:   factual
   Result:   PASS

4. What does a mark mean in AFL?
   Expected: factual
   Actual:   factual
   Result:   PASS

5. How many disposals did Ryan Abbott have in 2019?
   Expected: retrieval
   Actual:   retrieval
   Result:   PASS

6. What were Ryan Abbott's stats last round?
   Expected: retrieval
   Actual:   retrieval
   Result:   PASS

7. How many goals did Ryan Abbott score in 2019?
   Expected: retrieval
   Actual:   retrieval
   Result:   PASS

8. What is Geelong's record against Essendon?
   Expected: retrieval
   Actual:   retrieval
   Result:   PASS

9. How many matches have Geelong and Essendon played?
   Expected: retrieval
   Actual:   retrieval
   Result:   PASS

10. How many tackles did Ryan Abbott have?
   Ex

#### Accuracy Table

In [13]:
import pandas as pd

routing_df = pd.DataFrame(routing_results)

routing_accuracy = routing_df["Pass"].mean() * 100

print(f"Routing Accuracy: {routing_accuracy:.2f}%")

routing_df

Routing Accuracy: 100.00%


,Query,Expected,Actual,Pass
0,What are the basic rules of AFL?,factual,factual,True
1,What is a behind in AFL?,factual,factual,True
2,How many players are on an AFL team?,factual,factual,True
3,What does a mark mean in AFL?,factual,factual,True
4,How many disposals did Ryan Abbott have in 2019?,retrieval,retrieval,True
5,What were Ryan Abbott's stats last round?,retrieval,retrieval,True
6,How many goals did Ryan Abbott score in 2019?,retrieval,retrieval,True
7,What is Geelong's record against Essendon?,retrieval,retrieval,True
8,How many matches have Geelong and Essendon pla...,retrieval,retrieval,True
9,How many tackles did Ryan Abbott have?,retrieval,retrieval,True




The router node successfully classified all 20 test queries into the correct intent category.

The final routing accuracy was **100.00% (20/20)**, with no observed misroutes. Therefore, no further refinement of the router prompt or classification logic was necessary.

# **TASK 3**

## Wire Pridiction Model as Langraph Tool

### Team Alias Resolution

In [14]:
TEAM_ALIASES = {
    "pies": "Collingwood Magpies",
    "collingwood": "Collingwood Magpies",

    "cats": "Geelong Cats",
    "geelong": "Geelong Cats",

    "bombers": "Essendon Bombers",
    "essendon": "Essendon Bombers",

    "lions": "Brisbane Lions",
    "brisbane": "Brisbane Lions",

    "blues": "Carlton Blues",
    "carlton": "Carlton Blues",

    "crows": "Adelaide Crows",
    "adelaide": "Adelaide Crows",

    "dockers": "Fremantle Dockers",
    "fremantle": "Fremantle Dockers",

    "suns": "Gold Coast Suns",
    "gold coast": "Gold Coast Suns",

    "giants": "Greater Western Sydney Giants",
    "gws": "Greater Western Sydney Giants",

    "hawks": "Hawthorn Hawks",
    "hawthorn": "Hawthorn Hawks",

    "demons": "Melbourne Demons",
    "melbourne": "Melbourne Demons",

    "kangaroos": "North Melbourne Kangaroos",
    "north melbourne": "North Melbourne Kangaroos",

    "power": "Port Adelaide Power",
    "port adelaide": "Port Adelaide Power",

    "tigers": "Richmond Tigers",
    "richmond": "Richmond Tigers",

    "saints": "St Kilda Saints",
    "st kilda": "St Kilda Saints",

    "swans": "Sydney Swans",
    "sydney": "Sydney Swans",

    "eagles": "West Coast Eagles",
    "west coast": "West Coast Eagles",

    "bulldogs": "Western Bulldogs",
    "western bulldogs": "Western Bulldogs",
}

#### Normalize Names

In [15]:
def normalize_team_name(team: str) -> str:
    if not isinstance(team, str) or not team.strip():
        raise ValueError("Team name must be a non-empty string.")

    cleaned = team.strip().lower()

    if cleaned in TEAM_ALIASES:
        return TEAM_ALIASES[cleaned]

    canonical_teams = set(match_features["home_team"].dropna().unique()) | \
                      set(match_features["away_team"].dropna().unique())

    for canonical in canonical_teams:
        if cleaned == canonical.lower():
            return canonical

    raise ValueError(
        f"Unknown AFL team: '{team}'. "
        f"Use an AFL team name or supported nickname."
    )

#### Fixture Resolution

In [20]:
import pandas as pd


def resolve_fixture(team_a: str, team_b: str, match_date=None):
    team_a = normalize_team_name(team_a)
    team_b = normalize_team_name(team_b)

    if team_a == team_b:
        raise ValueError("The two teams must be different.")

    fixtures = match_features[
        (
            ((match_features["home_team"] == team_a) &
             (match_features["away_team"] == team_b))
            |
            ((match_features["home_team"] == team_b) &
             (match_features["away_team"] == team_a))
        )
    ].copy()

    if fixtures.empty:
        raise ValueError(
            f"No fixture found for {team_a} vs {team_b} "
            "in the available dataset."
        )

    fixtures["match_date"] = pd.to_datetime(fixtures["match_date"])

    if match_date is not None:
        requested_date = pd.to_datetime(match_date)

        exact = fixtures[
            fixtures["match_date"] == requested_date
        ]

        if exact.empty:
            raise ValueError(
                f"No {team_a} vs {team_b} fixture found on "
                f"{requested_date.date()}."
            )

        return exact.iloc[0]

    # If no date is supplied, use the latest fixture available
    # for these teams in the dataset.
    return fixtures.sort_values("match_date").iloc[-1]

#### Resolve "This Week"

In [21]:
def resolve_this_week_fixture(team_a: str, team_b: str):
    team_a = normalize_team_name(team_a)
    team_b = normalize_team_name(team_b)

    fixtures = match_features[
        (
            ((match_features["home_team"] == team_a) &
             (match_features["away_team"] == team_b))
            |
            ((match_features["home_team"] == team_b) &
             (match_features["away_team"] == team_a))
        )
    ].copy()

    if fixtures.empty:
        raise ValueError(
            f"No fixture data is available for {team_a} vs {team_b}."
        )

    fixtures["match_date"] = pd.to_datetime(fixtures["match_date"])

    today = pd.Timestamp.today().normalize()
    week_end = today + pd.Timedelta(days=6)

    this_week = fixtures[
        (fixtures["match_date"] >= today) &
        (fixtures["match_date"] <= week_end)
    ].sort_values("match_date")

    if this_week.empty:
        raise ValueError(
            f"No {team_a} vs {team_b} fixture is available for "
            f"the current week in the dataset."
        )

    return this_week.iloc[0]

In [118]:
graph_this_week_test = afl_graph.invoke({
    "user_query": "Who will be the top player for Geelong this week?",
    "conversation_history": []
})

print("Final response:")
print(graph_this_week_test["final_response"])

print("\nIntent:")
print(graph_this_week_test.get("intent"))

print("\nRoute:")
print(graph_this_week_test.get("route"))

print("\nValidation:")
print(graph_this_week_test.get("validation_result"))

print("\nNeeds clarification:")
print(graph_this_week_test.get("needs_clarification"))

print("\nTool result:")
print(graph_this_week_test.get("tool_results"))

Final response:
I couldn't resolve all the information needed for this request. Please provide a specific AFL team, player, or match date.

Intent:
prediction

Route:
prediction

Validation:
{'status': 'clarification', 'reason': 'no fixture for geelong cats is available this week in the available dataset.'}

Needs clarification:
True

Tool result:
{'error': 'No fixture for Geelong Cats is available this week in the available dataset.'}


#### Match Winner Prediction Tool

In [36]:
from langchain_core.tools import tool


@tool
def match_winner_prediction_tool(
    team_a: str,
    team_b: str,
    match_date: str = None
) -> dict:
    """
    Predict the winner of an AFL match.

    Accepts AFL team names or common nicknames such as Pies and Cats.
    Resolves the fixture and calls the Day 2 prediction model.
    """

    resolved_team_a = normalize_team_name(team_a)
    resolved_team_b = normalize_team_name(team_b)

    # Resolve fixture
    if match_date is None:
        fixture = resolve_fixture(
            resolved_team_a,
            resolved_team_b
        )
        resolved_date = fixture["match_date"]
    else:
        resolved_date = pd.to_datetime(match_date)

        # Make sure the requested fixture exists
        fixture = resolve_fixture(
            resolved_team_a,
            resolved_team_b,
            resolved_date
        )

    # Call Day 2 prediction model
    result = predict_match_winner(
        fixture["home_team"],
        fixture["away_team"],
        resolved_date
    )

    # Convert model class label into actual team name
    if result["winner"] == "Home Win":
        predicted_winner = fixture["home_team"]
    elif result["winner"] == "Away Win":
        predicted_winner = fixture["away_team"]
    else:
        predicted_winner = "Draw"

    return {
        "prediction_type": "match_winner",
        "home_team": fixture["home_team"],
        "away_team": fixture["away_team"],
        "match_date": str(pd.to_datetime(resolved_date).date()),
        "predicted_winner": predicted_winner,
        "probability": result["probability"],
        "class_probabilities": result["class_probabilities"]
    }

In [37]:
match_test = match_winner_prediction_tool.invoke({
    "team_a": "Pies",
    "team_b": "Cats"
})

match_test

{'prediction_type': 'match_winner',
 'home_team': 'Collingwood Magpies',
 'away_team': 'Geelong Cats',
 'match_date': '2025-05-03',
 'predicted_winner': 'Collingwood Magpies',
 'probability': 0.627,
 'class_probabilities': {'Away Win': 0.3669, 'Draw': 0.006, 'Home Win': 0.627}}

#### Top Player Prediction Tool

In [54]:
@tool
def top_player_prediction_tool(
    match_date: str,
    team: str,
    top_k: int = 5
) -> dict:
    """
    Predict the top AFL players for a team on a specified match date.

    Accepts AFL team names or common nicknames such as Cats and Pies.
    Uses the Day 2 predict_top_player function and includes
    available player-level grounding information.
    """
    resolved_team = normalize_team_name(team)

    try:
        resolved_date = pd.to_datetime(match_date).normalize()
    except Exception:
        raise ValueError(
            f"Invalid match date: '{match_date}'. "
            "Use a date such as '2025-09-27'."
        )

    result = predict_top_player(
        resolved_date,
        resolved_team,
        top_k=top_k
    )

    if not result:
        raise ValueError(
            f"No top-player prediction is available for "
            f"{resolved_team} on {resolved_date.date()}."
        )

    # Ensure notebook dataframe dates are normalized
    player_features["match_date"] = pd.to_datetime(
        player_features["match_date"],
        errors="coerce"
    ).dt.normalize()

    # Get players for the requested team and match date
    player_rows = player_features[
        (player_features["match_date"] == resolved_date) &
        (player_features["team"] == resolved_team)
    ].copy()

    if player_rows.empty:
        raise ValueError(
            f"Player feature data could not be found for "
            f"{resolved_team} on {resolved_date.date()}."
        )

    # Add available grounding information
    grounding = []

    for prediction in result:
        player_id = prediction["player_id"]

        matching_row = player_rows[
            player_rows["player_id"] == player_id
        ]

        if not matching_row.empty:
            recent_avg = matching_row.iloc[0][
                "player_recent_5_avg_disposals"
            ]

            grounding.append({
                "rank": prediction["rank"],
                "player_id": player_id,
                "recent_5_avg_disposals": (
                    round(float(recent_avg), 2)
                    if pd.notna(recent_avg)
                    else None
                ),
                "predicted_disposals": prediction["predicted_disposals"]
            })

    return {
        "prediction_type": "top_player",
        "team": resolved_team,
        "match_date": str(resolved_date.date()),
        "predicted_top_player": result[0],
        "ranked_predictions": result,
        "grounding_features": grounding
    }

In [44]:
player_features[
    player_features["team"] == "Geelong Cats"
][
    ["match_date", "team", "player_id"]
].sort_values("match_date").tail(10)

,match_date,team,player_id
68249,2025-09-27,Geelong Cats,44073
149177,2025-09-27,Geelong Cats,45000
13745,2025-09-27,Geelong Cats,43424
34254,2025-09-27,Geelong Cats,43674
216834,2025-09-27,Geelong Cats,45609
63992,2025-09-27,Geelong Cats,44030
149905,2025-09-27,Geelong Cats,45006
58174,2025-09-27,Geelong Cats,43957
21333,2025-09-27,Geelong Cats,43516
116107,2025-09-27,Geelong Cats,44617


In [55]:
top_player_test = top_player_prediction_tool.invoke({
    "match_date": "2025-09-27",
    "team": "Cats",
    "top_k": 5
})

top_player_test

{'prediction_type': 'top_player',
 'team': 'Geelong Cats',
 'match_date': '2025-09-27',
 'predicted_top_player': {'rank': 1,
  'player_id': 44960,
  'predicted_disposals': 27.67},
 'ranked_predictions': [{'rank': 1,
   'player_id': 44960,
   'predicted_disposals': 27.67},
  {'rank': 2, 'player_id': 44073, 'predicted_disposals': 23.44},
  {'rank': 3, 'player_id': 44487, 'predicted_disposals': 21.45},
  {'rank': 4, 'player_id': 43312, 'predicted_disposals': 21.19},
  {'rank': 5, 'player_id': 43674, 'predicted_disposals': 19.55}],
 'grounding_features': [{'rank': 1,
   'player_id': 44960,
   'recent_5_avg_disposals': 30.5,
   'predicted_disposals': 27.67},
  {'rank': 2,
   'player_id': 44073,
   'recent_5_avg_disposals': 24.8,
   'predicted_disposals': 23.44},
  {'rank': 3,
   'player_id': 44487,
   'recent_5_avg_disposals': 22.4,
   'predicted_disposals': 21.45},
  {'rank': 4,
   'player_id': 43312,
   'recent_5_avg_disposals': 22.0,
   'predicted_disposals': 21.19},
  {'rank': 5,
   'pl

In [111]:
def prediction_node(state: AFLState) -> dict:
    query = state["user_query"].lower()

    try:

        # 1. MATCH-WINNER PREDICTION
        if any(
            phrase in query
            for phrase in [
                "who will win",
                "who is likely to win",
                "predict the winner",
                "winner of",
                "win the match"
            ]
        ):

            team_mentions = []

            for alias, canonical in TEAM_ALIASES.items():
                position = query.find(alias)

                if position != -1:
                    team_mentions.append(
                        (position, canonical)
                    )

            team_mentions.sort(
                key=lambda x: x[0]
            )

            found_teams = []

            for _, canonical in team_mentions:
                if canonical not in found_teams:
                    found_teams.append(canonical)

            if len(found_teams) < 2:
                return {
                    "tool_results": {
                        "error": (
                            "Could not resolve two AFL teams "
                            "from the prediction request."
                        )
                    }
                }

            result = match_winner_prediction_tool.invoke({
                "team_a": found_teams[0],
                "team_b": found_teams[1]
            })

            return {
                "tool_results": result
            }


        # 2. TOP-PLAYER PREDICTION
        if any(
            phrase in query
            for phrase in [
                "top player",
                "top-score",
                "top scorer",
                "top score",
                "highest scorer",
                "most disposals"
            ]
        ):

            # Resolve team
            team_mentions = []

            for alias, canonical in TEAM_ALIASES.items():
                position = query.find(alias)

                if position != -1:
                    team_mentions.append(
                        (position, canonical)
                    )

            team_mentions.sort(
                key=lambda x: x[0]
            )

            found_teams = []

            for _, canonical in team_mentions:
                if canonical not in found_teams:
                    found_teams.append(canonical)

            if not found_teams:
                return {
                    "tool_results": {
                        "error": (
                            "Could not resolve an AFL team "
                            "from the prediction request."
                        )
                    }
                }

            resolved_team = found_teams[0]


            # Resolve explicit date if supplied
            import re

            date_match = re.search(
                r"\b(20\d{2}-\d{2}-\d{2})\b",
                query
            )

            if date_match:
                resolved_date = date_match.group(1)

            else:

                # "this week" handling
                if "this week" in query:

                    try:
                        from datetime import datetime, timedelta

                        today = pd.Timestamp.today().normalize()
                        week_end = today + pd.Timedelta(days=6)

                        team_rows = match_features[
                            (
                                (match_features["home_team"] == resolved_team) |
                                (match_features["away_team"] == resolved_team)
                            )
                        ].copy()

                        team_rows["match_date"] = pd.to_datetime(
                            team_rows["match_date"],
                            errors="coerce"
                        )

                        upcoming = team_rows[
                            (team_rows["match_date"] >= today) &
                            (team_rows["match_date"] <= week_end)
                        ].sort_values("match_date")

                        if upcoming.empty:
                            return {
                                "tool_results": {
                                    "error": (
                                        f"No fixture for {resolved_team} "
                                        "is available this week in the "
                                        "available dataset."
                                    )
                                }
                            }

                        resolved_date = str(
                            upcoming.iloc[0]["match_date"].date()
                        )

                    except Exception as e:
                        return {
                            "tool_results": {
                                "error": str(e)
                            }
                        }

                # No date supplied
                else:

                    try:
                        team_rows = match_features[
                            (
                                (match_features["home_team"] == resolved_team) |
                                (match_features["away_team"] == resolved_team)
                            )
                        ].copy()

                        team_rows["match_date"] = pd.to_datetime(
                            team_rows["match_date"],
                            errors="coerce"
                        )

                        team_rows = team_rows.dropna(
                            subset=["match_date"]
                        )

                        if team_rows.empty:
                            return {
                                "tool_results": {
                                    "error": (
                                        f"No fixture data is available "
                                        f"for {resolved_team}."
                                    )
                                }
                            }

                        # Use the latest available fixture in the dataset
                        resolved_date = str(
                            team_rows["match_date"].max().date()
                        )

                    except Exception as e:
                        return {
                            "tool_results": {
                                "error": str(e)
                            }
                        }


            # Call top-player prediction tool
            result = top_player_prediction_tool.invoke({
                "match_date": resolved_date,
                "team": resolved_team,
                "top_k": 5
            })

            return {
                "tool_results": result
            }


        # 3. UNSUPPORTED PREDICTION REQUEST
        return {
            "tool_results": {
                "error": (
                    "The prediction request could not be mapped "
                    "to a supported prediction tool."
                )
            }
        }


    except Exception as e:

        return {
            "tool_results": {
                "error": str(e)
            }
        }

In [70]:
prediction_test = prediction_node({
    "user_query": "Who will win Pies vs Cats?",
    "intent": "prediction"
})

prediction_test

{'tool_results': {'prediction_type': 'match_winner',
  'home_team': 'Collingwood Magpies',
  'away_team': 'Geelong Cats',
  'match_date': '2025-05-03',
  'predicted_winner': 'Collingwood Magpies',
  'probability': 0.627,
  'class_probabilities': {'Away Win': 0.3669,
   'Draw': 0.006,
   'Home Win': 0.627}}}

# **TASK 4**


In [82]:
from afl_retrieval_tools import (
    team_record_tool,
    player_season_stats_tool,
    TEAM_NAME_MAP,
    player_lookup
)

print("Retrieval tools and lookup data imported successfully.")

Retrieval tools and lookup data imported successfully.


### Retrival Node

In [135]:
def retrieval_node(state: AFLState) -> dict:
    query = state["user_query"].lower().strip()
    history = state.get("conversation_history", [])

    try:
        if history and any(
            word in query.split()
            for word in ["they", "them", "their", "that", "those"]
        ):
            previous_text = ""

            last_turn = history[-1]

            if isinstance(last_turn, dict):
                previous_text = (
                    str(last_turn.get("user", "")) +
                    " " +
                    str(last_turn.get("assistant", ""))
                ).lower()
            else:
                previous_text = str(last_turn).lower()

            previous_teams = []

            for alias, canonical in TEAM_NAME_MAP.items():
                position = previous_text.find(alias)

                if position != -1:
                    previous_teams.append(
                        (position, canonical)
                    )

            previous_teams.sort(key=lambda x: x[0])

            found_teams = []

            for _, canonical in previous_teams:
                if canonical not in found_teams:
                    found_teams.append(canonical)

            if len(found_teams) >= 2:
                team = found_teams[0]
                opponent = found_teams[1]

                result = team_record_tool.invoke({
                    "team": team,
                    "opponent": opponent
                })

                if "wins" in query or "win" in query:
                    return {
                        "tool_results": {
                            "follow_up_type": "wins",
                            "team": team,
                            "opponent": opponent,
                            "value": result["wins"]
                        }
                    }

                if "losses" in query or "loss" in query:
                    return {
                        "tool_results": {
                            "follow_up_type": "losses",
                            "team": team,
                            "opponent": opponent,
                            "value": result["losses"]
                        }
                    }

                if "draws" in query or "draw" in query:
                    return {
                        "tool_results": {
                            "follow_up_type": "draws",
                            "team": team,
                            "opponent": opponent,
                            "value": result["draws"]
                        }
                    }

                return {
                    "tool_results": result
                }

        player_stat_terms = [
            "stats",
            "statistics",
            "player stats",
            "player statistics",
            "season stats",
            "season statistics",
            "disposals",
            "kicks",
            "marks",
            "handballs",
            "goals",
            "tackles",
            "clearances",
            "inside 50",
            "inside_50"
        ]

        if any(term in query for term in player_stat_terms):
            matched_players = []

            for _, row in player_lookup.iterrows():
                player_name = str(
                    row["player_name"]
                ).strip()

                full_name = str(
                    row["player_full_name"]
                ).strip()

                if (
                    player_name.lower() in query
                    or full_name.lower() in query
                ):
                    matched_players.append(full_name)

            if not matched_players:
                return {
                    "tool_results": {
                        "error":
                            "Could not resolve a player from "
                            "the retrieval request."
                    }
                }

            player_name = matched_players[0]

            detected_year = None

            for year in range(1980, 2027):
                if str(year) in query:
                    detected_year = year
                    break

            result = player_season_stats_tool.invoke({
                "player_name": player_name,
                "year": detected_year
            })

            return {
                "tool_results": result
            }

        record_terms = [
            "record",
            "head to head",
            "head-to-head",
            "against",
            " vs ",
            "versus"
        ]

        if any(term in query for term in record_terms):
            team_mentions = []

            for alias, canonical in TEAM_NAME_MAP.items():
                position = query.find(alias)

                if position != -1:
                    team_mentions.append(
                        (position, canonical)
                    )

            team_mentions.sort(key=lambda x: x[0])

            found_teams = []

            for _, canonical in team_mentions:
                if canonical not in found_teams:
                    found_teams.append(canonical)

            if len(found_teams) < 2:
                return {
                    "tool_results": {
                        "error":
                            "Could not resolve two AFL teams "
                            "from the retrieval request."
                    }
                }

            result = team_record_tool.invoke({
                "team": found_teams[0],
                "opponent": found_teams[1]
            })

            return {
                "tool_results": result
            }

        return {
            "tool_results": {
                "error":
                    "The retrieval request could not be mapped "
                    "to a supported AFL retrieval tool."
            }
        }

    except Exception as e:
        return {
            "tool_results": {
                "error": str(e)
            }
        }


print("Clean retrieval_node loaded successfully.")

Clean retrieval_node loaded successfully.


In [93]:
retrieval_order_test = retrieval_node({
    "user_query": "What is Geelong's record against Essendon?",
    "intent": "retrieval"
})

print(retrieval_order_test)

{'tool_results': {'team': 'Geelong Cats', 'opponent': 'Essendon Bombers', 'matches': 63, 'wins': 39, 'losses': 23, 'draws': 1}}


In [84]:
retrieval_test_1 = retrieval_node({
    "user_query": "What is Geelong's record against Essendon?",
    "intent": "retrieval"
})

print(retrieval_test_1)

{'tool_results': {'team': 'Essendon Bombers', 'opponent': 'Geelong Cats', 'matches': 63, 'wins': 23, 'losses': 39, 'draws': 1}}


In [87]:
retrieval_test_2 = retrieval_node({
    "user_query": "What were Patrick Dangerfield's stats in 2025?",
    "intent": "retrieval"
})

print(retrieval_test_2)

{'tool_results': {'player': 'Patrick_Dangerfield', 'player_ids': [43674], 'year': 2025, 'records': [{'year': 2025, 'team': 'Geelong Cats', 'games_played': 20, 'kicks': 151.0, 'marks': 64.0, 'handballs': 139.0, 'disposals': 290.0, 'goals': 27.0, 'behinds': 22.0, 'tackles': 48.0, 'inside_50s': 60.0, 'clearances': 37.0, 'clangers': 55.0, 'rebound_50s': 2.0, 'one_percenters': 25.0}, {'year': 2025, 'team': 'Geelong Cats', 'games_played': 3, 'kicks': 29.0, 'marks': 13.0, 'handballs': 31.0, 'disposals': 60.0, 'goals': 3.0, 'behinds': nan, 'tackles': 9.0, 'inside_50s': 14.0, 'clearances': 12.0, 'clangers': 14.0, 'rebound_50s': 1.0, 'one_percenters': 5.0}]}}


### Validation Node

In [57]:
def validate_tool_result(state: AFLState) -> dict:
    """
    Validate retrieval/prediction results before the graph
    proceeds to the response formatter.
    """

    tool_results = state.get("tool_results")
    intent = state.get("intent")

    # 1. No result returned
    if tool_results is None:
        return {
            "validation_result": {
                "status": "fallback",
                "reason": "No tool result was returned."
            },
            "needs_clarification": False,
            "clarification_question": ""
        }

    # 2. Tool explicitly returned an error
    if isinstance(tool_results, dict) and tool_results.get("error"):

        error_message = str(
            tool_results["error"]
        ).lower()

        clarification_terms = [
            "unknown team",
            "unknown player",
            "no fixture",
            "no match found",
            "not found",
            "could not resolve",
            "invalid date",
            "outside the available data range"
        ]

        # Missing/ambiguous input → ask user
        if any(term in error_message for term in clarification_terms):
            return {
                "validation_result": {
                    "status": "clarification",
                    "reason": error_message
                },
                "needs_clarification": True,
                "clarification_question": (
                    "I couldn't resolve all the information needed "
                    "for this request. Please provide a specific "
                    "AFL team, player, or match date."
                )
            }

        # Other tool failure → fallback
        return {
            "validation_result": {
                "status": "fallback",
                "reason": error_message
            },
            "needs_clarification": False,
            "clarification_question": ""
        }

    # 3. Empty result
    if isinstance(tool_results, (dict, list)) and len(tool_results) == 0:
        return {
            "validation_result": {
                "status": "clarification",
                "reason": "The tool returned no usable data."
            },
            "needs_clarification": True,
            "clarification_question": (
                "I couldn't find enough AFL data to answer that. "
                "Could you provide more specific information?"
            )
        }

    # 4. Validate prediction-specific results
    if intent == "prediction" and isinstance(tool_results, dict):

        prediction_type = tool_results.get("prediction_type")

        # Match winner prediction
        if prediction_type == "match_winner":

            required_fields = [
                "predicted_winner",
                "probability",
                "match_date"
            ]

            missing_fields = [
                field
                for field in required_fields
                if field not in tool_results
            ]

            if missing_fields:
                return {
                    "validation_result": {
                        "status": "fallback",
                        "reason": (
                            "Match prediction is missing required "
                            f"fields: {missing_fields}"
                        )
                    },
                    "needs_clarification": False,
                    "clarification_question": ""
                }

        # Top-player prediction
        elif prediction_type == "top_player":

            ranked_predictions = tool_results.get(
                "ranked_predictions"
            )

            if not ranked_predictions:
                return {
                    "validation_result": {
                        "status": "clarification",
                        "reason": (
                            "No player prediction was returned."
                        )
                    },
                    "needs_clarification": True,
                    "clarification_question": (
                        "I couldn't find player prediction data "
                        "for that team and date. Please check "
                        "the team and match date."
                    )
                }


    # 5. Valid result
    return {
        "validation_result": {
            "status": "valid",
            "reason": "Tool result passed validation."
        },
        "needs_clarification": False,
        "clarification_question": ""
    }

In [58]:
validation_test = validate_tool_result({
    "intent": "prediction",
    "tool_results": top_player_test
})

validation_test

{'validation_result': {'status': 'valid',
  'reason': 'Tool result passed validation.'},
 'needs_clarification': False,
 'clarification_question': ''}

In [59]:
validation_error_test = validate_tool_result({
    "intent": "prediction",
    "tool_results": {
        "error": "Unknown team: 'ABC'"
    }
})

validation_error_test

{'validation_result': {'status': 'clarification',
  'reason': "unknown team: 'abc'"},
 'needs_clarification': True,
 'clarification_question': "I couldn't resolve all the information needed for this request. Please provide a specific AFL team, player, or match date."}

In [94]:
retrieval_valid_test = retrieval_node({
    "user_query": "What is Geelong's record against Essendon?",
    "intent": "retrieval"
})

validation_test = validate_tool_result({
    "user_query": "What is Geelong's record against Essendon?",
    "intent": "retrieval",
    "tool_results": retrieval_valid_test["tool_results"]
})

print("Retrieval result:")
print(retrieval_valid_test)

print("\nValidation result:")
print(validation_test)

Retrieval result:
{'tool_results': {'team': 'Geelong Cats', 'opponent': 'Essendon Bombers', 'matches': 63, 'wins': 39, 'losses': 23, 'draws': 1}}

Validation result:
{'validation_result': {'status': 'valid', 'reason': 'Tool result passed validation.'}, 'needs_clarification': False, 'clarification_question': ''}


### Conditional Validation Routig

In [60]:
def route_after_validation(state: AFLState) -> str:
    """
    Decide where the graph goes after validation.
    """

    validation = state.get("validation_result", {})
    status = validation.get("status")

    if status == "valid":
        return "response_formatter"

    if status == "clarification":
        return "clarification"

    if status == "fallback":
        return "fallback"

    # Defensive fallback if validation status is unexpected
    return "fallback"

### Clarification Node

In [61]:
def clarification_node(state: AFLState) -> dict:
    """
    Return the clarification question generated by validation.
    """

    question = state.get(
        "clarification_question",
        "Could you provide more specific AFL information?"
    )

    return {
        "final_response": question
    }

### Fallback Node

In [62]:
def fallback_node(state: AFLState) -> dict:
    """
    Handle unsupported, ambiguous, or failed requests
    without hallucinating an answer.
    """

    return {
        "final_response": (
            "I couldn't answer that using the available AFL "
            "data and prediction models. I can currently handle "
            "AFL factual/retrieval questions, match-winner "
            "predictions, and top-player predictions."
        )
    }

In [63]:
clarification_test = clarification_node({
    "clarification_question": (
        "Which AFL team did you mean? "
        "Please provide the team name."
    )
})

fallback_test = fallback_node({})

print("Clarification:")
print(clarification_test)

print("\nFallback:")
print(fallback_test)

Clarification:
{'final_response': 'Which AFL team did you mean? Please provide the team name.'}

Fallback:
{'final_response': "I couldn't answer that using the available AFL data and prediction models. I can currently handle AFL factual/retrieval questions, match-winner predictions, and top-player predictions."}


In [64]:
# Test 1: Valid result
valid_route = route_after_validation({
    "validation_result": {
        "status": "valid"
    }
})

# Test 2: Clarification required
clarification_route = route_after_validation({
    "validation_result": {
        "status": "clarification"
    }
})

# Test 3: Fallback required
fallback_route = route_after_validation({
    "validation_result": {
        "status": "fallback"
    }
})

# Test 4: Unexpected status
unexpected_route = route_after_validation({
    "validation_result": {
        "status": "something_else"
    }
})

print("Valid route:", valid_route)
print("Clarification route:", clarification_route)
print("Fallback route:", fallback_route)
print("Unexpected status route:", unexpected_route)

Valid route: response_formatter
Clarification route: clarification
Fallback route: fallback
Unexpected status route: fallback


### Direct Answer Node

In [95]:
def direct_answer_node(state: AFLState) -> dict:
    query = state["user_query"]

    prompt = f"""
You are an AFL-only assistant.

Answer the user's question using general AFL knowledge.

Rules:
- Stay strictly within Australian Football League (AFL) topics.
- Do not invent statistics, match results, player statistics, or predictions.
- If the question requires dataset-specific statistics, say that it
  should be handled by the retrieval system.
- Keep the answer concise and clear.

User question:
{query}
"""

    try:
        response = llm.invoke(prompt)

        return {
            "final_response": response.content
        }

    except Exception as e:
        return {
            "final_response": (
                "I couldn't generate an AFL factual answer "
                "right now."
            ),
            "tool_results": {
                "error": str(e)
            }
        }

In [96]:
factual_test = direct_answer_node({
    "user_query": "What is a behind in AFL?",
    "intent": "factual"
})

print(factual_test)

{'final_response': 'In AFL, a behind is a score worth one point. It occurs when the ball is kicked between the goalposts but does not pass between the two taller goalposts, or if the ball is touched by a player before crossing the goal line. A behind can also be awarded if the ball goes out of bounds after being last touched by an attacking player.'}


### Refusal Node

In [97]:
def refusal_node(state: AFLState) -> dict:
    return {
        "final_response": (
            "I'm an AFL-focused assistant, so I can help with "
            "AFL teams, players, matches, statistics, history, "
            "rules, and AFL predictions. I can't answer "
            "questions outside AFL."
        )
    }

In [98]:
refusal_test = refusal_node({
    "user_query": "What is the capital of France?",
    "intent": "off-topic"
})

print(refusal_test)

{'final_response': "I'm an AFL-focused assistant, so I can help with AFL teams, players, matches, statistics, history, rules, and AFL predictions. I can't answer questions outside AFL."}


In [109]:
graph_clarification_test = afl_graph.invoke({
    "user_query": "Who will win Tigers vs Unknown FC?",
    "conversation_history": []
})

print("Final response:")
print(graph_clarification_test["final_response"])

print("\nIntent:")
print(graph_clarification_test.get("intent"))

print("\nRoute:")
print(graph_clarification_test.get("route"))

print("\nValidation:")
print(graph_clarification_test.get("validation_result"))

print("\nNeeds clarification:")
print(graph_clarification_test.get("needs_clarification"))

print("\nTool result:")
print(graph_clarification_test.get("tool_results"))

Final response:
I couldn't resolve all the information needed for this request. Please provide a specific AFL team, player, or match date.

Intent:
prediction

Route:
prediction

Validation:
{'status': 'clarification', 'reason': 'could not resolve two afl teams from the prediction request.'}

Needs clarification:
True

Tool result:
{'error': 'Could not resolve two AFL teams from the prediction request.'}


In [136]:
def response_formatter(state: AFLState) -> dict:
    tool_results = state.get("tool_results", {})
    intent = state.get("intent")

    if intent == "retrieval":

        if tool_results.get("follow_up_type"):
            follow_up_type = tool_results["follow_up_type"]
            team = tool_results["team"]
            opponent = tool_results["opponent"]
            value = tool_results["value"]

            if follow_up_type == "wins":
                response = (
                    f"{team} had {value} wins against "
                    f"{opponent}."
                )

            elif follow_up_type == "losses":
                response = (
                    f"{team} had {value} losses against "
                    f"{opponent}."
                )

            elif follow_up_type == "draws":
                draw_word = (
                    "draw"
                    if value == 1
                    else "draws"
                )

                response = (
                    f"{team} had {value} {draw_word} against "
                    f"{opponent}."
                )

            else:
                response = str(tool_results)

            return {
                "final_response": response
            }

        if (
            "team" in tool_results
            and "opponent" in tool_results
        ):
            team = tool_results["team"]
            opponent = tool_results["opponent"]

            matches = tool_results.get("matches", 0)
            wins = tool_results.get("wins", 0)
            losses = tool_results.get("losses", 0)
            draws = tool_results.get("draws", 0)

            if team.endswith("s"):
                team_possessive = f"{team}'"
            else:
                team_possessive = f"{team}'s"

            draw_word = (
                "draw"
                if draws == 1
                else "draws"
            )

            return {
                "final_response": (
                    f"{team_possessive} record against "
                    f"{opponent} is {wins} wins, "
                    f"{losses} losses, and {draws} "
                    f"{draw_word} across {matches} matches."
                )
            }

        if (
            "player" in tool_results
            and "records" in tool_results
        ):
            player = str(
                tool_results["player"]
            ).replace("_", " ")

            year = tool_results.get("year")
            records = tool_results.get("records", [])

            if year:
                response = (
                    f"Retrieved {len(records)} record(s) "
                    f"for {player} in {year}."
                )
            else:
                response = (
                    f"Retrieved {len(records)} record(s) "
                    f"for {player}."
                )

            return {
                "final_response": response
            }

    if intent == "prediction":

        if (
            tool_results.get("prediction_type")
            == "match_winner"
        ):
            winner = tool_results["predicted_winner"]
            probability = tool_results["probability"]
            home = tool_results["home_team"]
            away = tool_results["away_team"]
            match_date = tool_results["match_date"]

            class_probs = tool_results.get(
                "class_probabilities", {}
            )

            grounding = get_match_grounding_features(
                home,
                away,
                match_date
            )

            probability_text = (
                f"{probability * 100:.1f}%"
            )

            response = (
                f"Model prediction for {home} vs {away} "
                f"on {match_date}:\n\n"
                f"Predicted winner: {winner}\n"
                f"Model probability: {probability_text}\n"
            )

            if class_probs:
                response += "\nClass probabilities:\n"

                for label, value in class_probs.items():
                    response += (
                        f"- {label}: "
                        f"{value * 100:.1f}%\n"
                    )

            if grounding:
                response += (
                    "\nSelected grounding inputs:\n"
                )

                for feature, value in grounding:
                    response += (
                        f"- {feature}: {value}\n"
                    )

            response += (
                "\nThis is a model prediction, "
                "not a certainty."
            )

            return {
                "final_response": response
            }

        if (
            tool_results.get("prediction_type")
            == "top_player"
        ):
            team = tool_results["team"]
            match_date = tool_results["match_date"]

            top_player = tool_results[
                "predicted_top_player"
            ]

            player_id = top_player["player_id"]

            predicted_disposals = top_player[
                "predicted_disposals"
            ]

            response = (
                f"Top-player model prediction for "
                f"{team} on {match_date}:\n\n"
                f"Predicted top player ID: "
                f"{player_id}\n"
                f"Predicted disposals: "
                f"{predicted_disposals:.2f}\n\n"
                f"The current top-player model does not "
                f"provide a calibrated probability, so "
                f"no probability has been invented. "
                f"This is a model prediction, "
                f"not a certainty."
            )

            grounding = tool_results.get(
                "grounding_features", []
            )

            if grounding:
                first = grounding[0]

                recent_avg = first.get(
                    "recent_5_avg_disposals"
                )

                if recent_avg is not None:
                    response += (
                        "\n\nSelected grounding input: "
                        "recent 5-match average "
                        f"disposals = {recent_avg:.2f}."
                    )

            return {
                "final_response": response
            }

    return {
        "final_response": str(tool_results)
    }


print("Clean response_formatter loaded successfully.")

Clean response_formatter loaded successfully.


# **TASK 5**

### Graph

In [138]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AFLState)

workflow.add_node("router", router_node)
workflow.add_node("direct_answer", direct_answer_node)
workflow.add_node("retrieval", retrieval_node)
workflow.add_node("prediction", prediction_node)
workflow.add_node("refusal", refusal_node)
workflow.add_node("validation", validate_tool_result)
workflow.add_node("response_formatter", response_formatter)
workflow.add_node("clarification", clarification_node)
workflow.add_node("fallback", fallback_node)

workflow.add_edge(START, "router")

workflow.add_conditional_edges(
    "router",
    route_from_router,
    {
        "direct_answer": "direct_answer",
        "retrieval": "retrieval",
        "prediction": "prediction",
        "refusal": "refusal"
    }
)

workflow.add_edge("direct_answer", END)
workflow.add_edge("refusal", END)

workflow.add_edge("retrieval", "validation")
workflow.add_edge("prediction", "validation")

workflow.add_conditional_edges(
    "validation",
    route_after_validation,
    {
        "response_formatter": "response_formatter",
        "clarification": "clarification",
        "fallback": "fallback"
    }
)

workflow.add_edge("response_formatter", END)
workflow.add_edge("clarification", END)
workflow.add_edge("fallback", END)

afl_graph = workflow.compile()

print("Graph recompiled successfully.")

Graph recompiled successfully.


### End to End Retrival Test

In [104]:
graph_retrieval_test = afl_graph.invoke({
    "user_query": "What is Geelong's record against Essendon?",
    "conversation_history": []
})

print("Final response:")
print(graph_retrieval_test["final_response"])

print("\nIntent:")
print(graph_retrieval_test.get("intent"))

print("\nRoute:")
print(graph_retrieval_test.get("route"))

print("\nValidation:")
print(graph_retrieval_test.get("validation_result"))

Final response:
Geelong Cats's record against Essendon Bombers is 39 wins, 23 losses, and 1 draws across 63 matches.

Intent:
retrieval

Route:
retrieval

Validation:
{'status': 'valid', 'reason': 'Tool result passed validation.'}


In [124]:
graph_retrieval_test = afl_graph.invoke({
    "user_query": "What is Geelong's record against Essendon?",
    "conversation_history": []
})

print(graph_retrieval_test["final_response"])

Geelong Cats's record against Essendon Bombers is 39 wins, 23 losses, and 1 draws across 63 matches.


In [108]:
graph_prediction_test = afl_graph.invoke({
    "user_query": "Who will win Pies vs Cats?",
    "conversation_history": []
})

print("Final response:")
print(graph_prediction_test["final_response"])

print("\nIntent:")
print(graph_prediction_test.get("intent"))

print("\nRoute:")
print(graph_prediction_test.get("route"))

print("\nValidation:")
print(graph_prediction_test.get("validation_result"))

print("\nTool result:")
print(graph_prediction_test.get("tool_results"))

Final response:
Model prediction for Collingwood Magpies vs Geelong Cats on 2025-05-03:

Predicted winner: Collingwood Magpies
Model probability: 62.7%

Class probabilities:
- Away Win: 36.7%
- Draw: 0.6%
- Home Win: 62.7%


Selected grounding inputs:
- home_recent_5_win_rate: 1.0
- away_recent_5_win_rate: 0.6
- home_recent_5_avg_score: 92.0

This is a model prediction, not a certainty.

Intent:
prediction

Route:
prediction

Validation:
{'status': 'valid', 'reason': 'Tool result passed validation.'}

Tool result:
{'prediction_type': 'match_winner', 'home_team': 'Collingwood Magpies', 'away_team': 'Geelong Cats', 'match_date': '2025-05-03', 'predicted_winner': 'Collingwood Magpies', 'probability': 0.627, 'class_probabilities': {'Away Win': 0.3669, 'Draw': 0.006, 'Home Win': 0.627}}


In [114]:
graph_top_player_test = afl_graph.invoke({
    "user_query": "Who will be the top player for Geelong?",
    "conversation_history": []
})

print("Final response:")
print(graph_top_player_test["final_response"])

print("\nIntent:")
print(graph_top_player_test.get("intent"))

print("\nRoute:")
print(graph_top_player_test.get("route"))

print("\nValidation:")
print(graph_top_player_test.get("validation_result"))

print("\nTool result:")
print(graph_top_player_test.get("tool_results"))

Final response:
Top-player model prediction for Geelong Cats on 2025-09-27:

Predicted top player ID: 44960
Predicted disposals: 27.67

The current top-player model does not provide a calibrated probability, so no probability has been invented. This is a model prediction, not a certainty.

Selected grounding input: recent 5-match average disposals = 30.50.

Intent:
prediction

Route:
prediction

Validation:
{'status': 'valid', 'reason': 'Tool result passed validation.'}

Tool result:
{'prediction_type': 'top_player', 'team': 'Geelong Cats', 'match_date': '2025-09-27', 'predicted_top_player': {'rank': 1, 'player_id': 44960, 'predicted_disposals': 27.67}, 'ranked_predictions': [{'rank': 1, 'player_id': 44960, 'predicted_disposals': 27.67}, {'rank': 2, 'player_id': 44073, 'predicted_disposals': 23.44}, {'rank': 3, 'player_id': 44487, 'predicted_disposals': 21.45}, {'rank': 4, 'player_id': 43312, 'predicted_disposals': 21.19}, {'rank': 5, 'player_id': 43674, 'predicted_disposals': 19.55}],

# **TASK 5**

### End to End testing

In [120]:
test_cases = [
    {
        "id": 1,
        "query": "What is a behind in AFL?",
        "category": "factual"
    },
    {
        "id": 2,
        "query": "What are the basic rules of AFL?",
        "category": "factual"
    },
    {
        "id": 3,
        "query": "What is Geelong's record against Essendon?",
        "category": "retrieval"
    },
    {
        "id": 4,
        "query": "What are Geelong's player stats?",
        "category": "retrieval"
    },
    {
        "id": 5,
        "query": "Who will win Pies vs Cats?",
        "category": "match prediction"
    },
    {
        "id": 6,
        "query": "Who will be the top player for Geelong on 2025-09-27?",
        "category": "player prediction"
    },
    {
        "id": 7,
        "query": "What is the capital of France?",
        "category": "off-topic"
    },
    {
        "id": 8,
        "query": "Who will win Tigers vs Unknown FC?",
        "category": "ambiguous prediction"
    },
    {
        "id": 9,
        "query": "Who will be the top player for Geelong this week?",
        "category": "unavailable fixture"
    }
]

results = []

for test in test_cases:

    try:
        result = afl_graph.invoke({
            "user_query": test["query"],
            "conversation_history": []
        })

        results.append({
            "test_id": test["id"],
            "category": test["category"],
            "query": test["query"],
            "intent": result.get("intent"),
            "route": result.get("route"),
            "validation_status": (
                result.get("validation_result", {}).get("status")
                if result.get("validation_result")
                else "N/A"
            ),
            "needs_clarification": result.get(
                "needs_clarification",
                False
            ),
            "final_response": result.get(
                "final_response",
                ""
            )
        })

    except Exception as e:

        results.append({
            "test_id": test["id"],
            "category": test["category"],
            "query": test["query"],
            "intent": "ERROR",
            "route": "ERROR",
            "validation_status": "ERROR",
            "needs_clarification": False,
            "final_response": str(e)
        })


import pandas as pd

task5_results = pd.DataFrame(results)

display(task5_results[
    [
        "test_id",
        "category",
        "query",
        "intent",
        "route",
        "validation_status",
        "needs_clarification"
    ]
])

,test_id,category,query,intent,route,validation_status,needs_clarification
0,1,factual,What is a behind in AFL?,factual,direct_answer,N/A,False
1,2,factual,What are the basic rules of AFL?,factual,direct_answer,N/A,False
2,3,retrieval,What is Geelong's record against Essendon?,retrieval,retrieval,valid,False
3,4,retrieval,What are Geelong's player stats?,retrieval,retrieval,clarification,True
4,5,match prediction,Who will win Pies vs Cats?,prediction,prediction,valid,False
5,6,player prediction,Who will be the top player for Geelong on 2025...,prediction,prediction,valid,False
6,7,off-topic,What is the capital of France?,off-topic,refusal,N/A,False
7,8,ambiguous prediction,Who will win Tigers vs Unknown FC?,prediction,prediction,clarification,True
8,9,unavailable fixture,Who will be the top player for Geelong this week?,prediction,prediction,clarification,True


In [121]:
for _, row in task5_results.iterrows():

    print("=" * 80)
    print(f"TEST {row['test_id']} — {row['category']}")
    print(f"QUERY: {row['query']}")
    print(f"INTENT: {row['intent']}")
    print(f"ROUTE: {row['route']}")
    print(f"VALIDATION: {row['validation_status']}")
    print(f"CLARIFICATION: {row['needs_clarification']}")
    print("\nFINAL RESPONSE:")
    print(row["final_response"])

TEST 1 — factual
QUERY: What is a behind in AFL?
INTENT: factual
ROUTE: direct_answer
VALIDATION: N/A
CLARIFICATION: False

FINAL RESPONSE:
In AFL, a behind is a score worth one point. It occurs when the ball is kicked between the goalposts but does not pass between the two taller goalposts, or if the ball is touched by a player before crossing the goal line. A behind can also be awarded if the ball goes out of bounds after being last touched by an attacking player.
TEST 2 — factual
QUERY: What are the basic rules of AFL?
INTENT: factual
ROUTE: direct_answer
VALIDATION: N/A
CLARIFICATION: False

FINAL RESPONSE:
The basic rules of AFL (Australian Football League) include:

1. **Field and Teams**: The game is played on an oval field with two teams of 18 players each.

2. **Scoring**: Points are scored by kicking the ball between the goalposts. A goal (6 points) is scored when the ball is kicked through the two taller middle posts, while a behind (1 point) is scored if the ball passes bet

### Multi-turn Test

In [139]:
conversation_history = []

# Turn 1
turn_1 = afl_graph.invoke({
    "user_query": "What is Geelong's record against Essendon?",
    "conversation_history": conversation_history
})

conversation_history.append({
    "user": "What is Geelong's record against Essendon?",
    "assistant": turn_1["final_response"]
})

# Turn 2
turn_2 = afl_graph.invoke({
    "user_query": "How many wins did they have?",
    "conversation_history": conversation_history
})

print("TURN 1")
print("Intent:", turn_1.get("intent"))
print("Route:", turn_1.get("route"))
print("Response:", turn_1["final_response"])

print("\n" + "=" * 80)

print("TURN 2")
print("Intent:", turn_2.get("intent"))
print("Route:", turn_2.get("route"))
print("Response:", turn_2["final_response"])

TURN 1
Intent: retrieval
Route: retrieval
Response: Geelong Cats' record against Essendon Bombers is 39 wins, 23 losses, and 1 draw across 63 matches.

TURN 2
Intent: retrieval
Route: retrieval
Response: Geelong Cats had 39 wins against Essendon Bombers.


## Full State Trace

### Trace 1 Match Prediction

1. **Query:** Who will win Pies vs Cats?
2. **Router Decision:**
     * Intent: prediction
     * Route: prediction
3. **Tool Called:**  match_winner_prediction_tool
     * Input:
       * team_a = Collingwood Magpies
        * team_b = Geelong Cats

4. **Validation:**
     * Status: valid
     * Reason: Tool result passed validation.

5. **Final Response:**

" Model prediction for Collingwood Magpies vs Geelong Cats on 2025-05-03:

Predicted winner: Collingwood Magpies
Model probability: 62.7%

Class probabilities:
- Away Win: 36.7%
- Draw: 0.6%
- Home Win: 62.7%


Selected grounding inputs:
- home_recent_5_win_rate: 1.0
- away_recent_5_win_rate: 0.6
- home_recent_5_avg_score: 92.0

This is a model prediction, not a certainty."

**Annotation:** The router recognized "Who will win" as a prediction request and routed it to the prediction node. Pies and Cats were resolved to the canonical dataset names and the Day 2 prediction model was called. The required prediction fields were present, so the graph continued to the response formatter. The formatter converted the structured prediction result into a readable response and included probability and uncertainty.


---


      

### Trace 2 Retrival

1. **Query:** What is Geelong's record against Essendon?
2. **Router Decision:**
     * Intent: retrival
     * Route: retrival
3. **Tool Called:**  team_record_tool
     * Input:
       * team_a = Geelong Cats
        * team_b = Essendon Bombers
        Matches: 63
Wins: 39
Losses: 23
Draws: 1

4. **Validation:**
     * Status: valid
     * Reason: Tool result passed validation.

5. **Final Response:**  Geelong Cats's record against Essendon Bombers is 39 wins, 23 losses, and 1 draws across 63 matches.

**Annotation:** This is an exact statistical/data question, so the router selected the retrieval path. The tool retrieved the numbers directly from the structured AFL dataset rather than asking the LLM to invent them. The retrieval result contained usable data, so the graph proceeded to the response formatter. The formatter converted the structured dictionary into a concise natural-language answer.


---




### Trace 3 Ambiguous Prediction

1. **Query:** Who will win Tigers vs Unknown FC?

2. **Router Decision:**
   - Intent: prediction
   - Route: prediction

3. **Tool Called:** match_winner_prediction_tool
   - Input:
     - team_a = Richmond Tigers
     - team_b = Unknown FC
   - Tigers → Richmond Tigers
   - Unknown FC → unresolved
   - Error: Could not resolve two AFL teams from the prediction request.

4. **Validation:**
   - Status: clarification
   - Reason: Could not resolve two AFL teams from the prediction request.
   - Needs clarification: True

5. **Final Response:** I couldn't resolve all the information needed for this request. Please provide a specific AFL team, player, or match date.

**Annotation:** The router identified the query as a prediction request and routed it to the prediction node. The system successfully resolved Tigers to the canonical team name Richmond Tigers, but Unknown FC could not be matched to an AFL team in the dataset. Instead of guessing the missing team, the prediction tool returned an error. The validation node detected the unresolved information and changed the result to clarification. The graph therefore routed the request to the clarification node, which asked the user to provide specific AFL information.

## LangGraph orchestration vs Single monolithic LangChain agent
LangGraph improved the system by separating routing, retrieval, prediction, validation, clarification, and response formatting into explicit nodes, making the workflow more predictable and easier to debug. Compared with a single monolithic LangChain agent, it also provides stronger control over prediction requests, ensuring model outputs are validated, uncertainty is communicated, and ambiguous inputs trigger clarification instead of unsupported answers.

### Routing Accuracy Evaluation

The router was evaluated using 15 AFL and non-AFL queries covering factual questions, retrieval questions, prediction requests, and off-topic requests.

| # | Query | Expected Intent | Predicted Intent | Correct |
|---|---|---|---|---|
| 1 | What is a behind in AFL? | factual | factual | Yes |
| 2 | What are the basic rules of AFL? | factual | factual | Yes |
| 3 | What is a mark in AFL? | factual | factual | Yes |
| 4 | What is a handball in AFL? | factual | factual | Yes |
| 5 | What is Geelong's record against Essendon? | retrieval | retrieval | Yes |
| 6 | How many wins does Geelong have against Essendon? | retrieval | retrieval | Yes |
| 7 | What are Geelong's player stats? | retrieval | retrieval | Yes |
| 8 | How many goals did a player score? | retrieval | retrieval | Yes |
| 9 | Who will win Pies vs Cats? | prediction | prediction | Yes |
| 10 | Who is likely to win Richmond vs Carlton? | prediction | prediction | Yes |
| 11 | Who will be the top player for Geelong? | prediction | prediction | Yes |
| 12 | Who will top-score in the match? | prediction | prediction | Yes |
| 13 | What is the capital of France? | off-topic | off-topic | Yes |
| 14 | What is today's weather? | off-topic | off-topic | Yes |
| 15 | Tell me a Python joke. | off-topic | off-topic | Yes |

**Routing Accuracy:**

\[
\text{Accuracy} =
\frac{\text{Number of Correct Predictions}}
{\text{Total Test Cases}}
\times 100
\]

**Result:**

15 out of 15 test cases were routed to the expected intent.

**Routing Accuracy = 100%**